In [1]:
%run ~/analysisLibraries

In [2]:
%matplotlib inline

In [3]:
%load_ext rpy2.ipython

Read data and make SDA tables. Either read the separate datasets and concatenate them, or read the data concatenated as a whole.

In [8]:
CELL = sc.read('../../../wholecell_samuele/write/qc/DGE_04.h5ad')

In [9]:
NUCLEI = sc.read('../../write/MERGED_normalized_04.h5ad')

In [12]:
NUCLEI.X = NUCLEI.layers['norm_sct'].copy()
sc.pp.scale(NUCLEI)

In [13]:
CELL.X = CELL.layers['norm_sct'].copy()
sc.pp.scale(CELL)

In [ ]:
names = np.intersect1d(CELL.var_names, NUCLEI.var_names)
CELL = CELL[:,names].copy()
NUCLEI = NUCLEI[:,names].copy()

Make dataframes and concatenate. You can eventually only choose HVGs ore keep all the genes as I did

In [14]:
dfCELL = pd.DataFrame(CELL.X, 
                        index=CELL.obs_names,
                        columns=CELL.var_names)
dfNUCLEI = pd.DataFrame(NUCLEI.X, 
                        index=NUCLEI.obs_names,
                        columns=NUCLEI.var_names)

In [ ]:
dfDATA = pd.concat([dfCELL,dfNUCLEI])

export data and cells barcodes

In [20]:
dfDATA.shape

(10522, 14409)

In [21]:
dfDATA.to_csv('./ALLDATA_scaled_onematrix_data.csv' ,
            index=False, header=False, sep=' ')

Now you need to run the script for sda `sdaWorkflow.py` with `gwf`. You need to write by hand the number of cells in the script. It is highlighted by a comment.

In [22]:
!gwf -b slurm -f sdaWorkflow.py run ALLDATA_scaled_onematrix_data

You are currently running version 1.7.1, but the latest version is 1.7.2. Consider updating. You can disable this check with `gwf config set check_updates no`.


Analysis Example on one component

In [5]:
adata = ad.AnnData.concatenate(CELL, NUCLEI)

In [7]:
A = pd.read_csv('./ALLDATA_scaled_onematrix_data/it2000/A', sep=' ', header=None)
#B is only for multi-tissues tensor
#B = pd.read_csv('SDA/ALLDATA_scaled_onematrix_data/it2000/B1', sep=' ', header=None)
X = pd.read_csv('./ALLDATA_scaled_onematrix_data/it2000/X1', sep=' ', header=None)

$A$ contains the score of each cell in the components. This score can be used as a "soft clustering" once it is normalized. We will try to do that in the notebook that follows this one. 

In [8]:
A.shape

(10522, 15)

In [9]:
A.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.823578,-0.150856,-0.232868,0.978262,0.941127,0.149279,-0.895419,0.488612,-0.142367,-0.650650,0.047764,-1.797530,-0.291950,-0.285636,-0.051455
1,0.412280,-0.381873,-0.153474,-0.479830,-0.742368,0.240343,0.763881,0.456313,0.191713,-0.217358,-0.112295,0.387332,-0.520114,-0.011430,-1.238040
2,-0.116371,-0.155019,0.157816,-1.017400,0.526689,0.398200,-0.821739,0.538557,-0.649574,-0.507511,0.035280,0.748228,2.085870,0.128419,0.152133
3,-0.191306,-0.301525,2.223640,-0.386508,-0.229589,0.379063,-0.091058,0.557750,-0.022447,1.450060,-0.230327,0.628984,-1.147190,-0.250618,1.056060
4,0.662584,-0.424341,-0.118680,-0.645929,-0.383942,0.136237,0.043829,-0.638192,-0.008545,-0.352948,-0.009485,0.573066,-0.846059,-0.054262,-0.036807


$X$ contains the loading of each gene in each component according to the spike-and-slab prior. The sign of the loading has to be considered together with the sign of the cell score to see if the resulting sign is positive (significant expression of gene) or negative (significant underexpression of gene)

In [10]:
X.columns = adata.var_names

In [11]:
X.shape

(15, 14409)

In [12]:
X.head()

index,A1BG,A2M,A2ML1,AAAS,AACS,AADAT,AAED1,AAGAB,AAK1,AAMDC,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
0,-0.001271,0.002861,-0.040488,0.000411,0.000119,-0.000019,-0.109392,-0.035660,-0.000174,-0.101341,...,-0.000968,-0.000006,0.000177,-0.000106,-0.000019,-0.000268,0.000096,-0.000114,0.000470,-0.000680
1,0.003783,0.113529,0.038002,-0.000122,-0.000143,0.069189,-0.000034,-0.073289,-0.079646,-0.000039,...,0.000427,-0.014813,-0.007363,-0.001254,0.140359,-0.000068,-0.032127,-0.037927,-0.000106,-0.035350
2,-0.000145,-0.001594,0.000110,-0.000199,-0.000277,0.000139,-0.000031,0.084301,0.000007,0.001438,...,-0.003909,0.056287,0.000106,0.006375,0.000083,0.000129,0.032034,-0.000221,-0.004367,-0.030715
3,-0.000040,0.038087,-0.085117,-0.000052,-0.000249,-0.000071,0.000782,0.012083,0.006578,-0.127971,...,-0.000163,-0.000272,-0.000206,-0.043670,0.000392,0.000030,-0.000049,0.000167,-0.059943,-0.000244
4,-0.000249,-0.000555,0.000264,0.000464,-0.030960,0.005369,-0.000225,-0.042875,0.043919,-0.000871,...,-0.000334,-0.000312,0.071707,-0.001734,-0.000366,-0.000091,0.000099,0.000568,0.062290,-0.001858


## Calculate already some stuff to save it in the annotated data object
## Also for the 15 components decomposition

All components' cell scores 

In [13]:
for i in range(X.shape[0]):
    adata.obs['SDA_cellscore_C'+str(i)] = np.asarray(A.iloc[:,i])

All components' significant cells. I chose threshold based on different runs of SDA and looking at how scores looked like in plots. For each component a cell will be irrelevant, negative or positive based on score thresholds.

In [14]:
A.index = adata.obs_names
for i in range(X.shape[0]):
    score_neg = min(-1.5, np.quantile(A[i], .1))
    score_pos = max(+1.5, np.quantile(A[i], .9))
    cells_pos = A[i][ A[i]>score_pos ].index
    cells_neg = A[i][ A[i]<score_neg ].index
    adata.obs['sda_cell_relevant_C'+str(i)] = pd.Categorical(
        ['POSITIVE' if i in cells_pos 
         else 'NEGATIVE' if i in cells_neg
         else 'IRRELEVANT'
         for i in adata.obs_names])

All components' significant genes. Significant genes for positive and negative components are saved with a boolean vector in the .var storage of the adata object. Relevant genes are saved with a relaxed and strict threshold (high and low in the name). 

In [15]:
for i in range(X.shape[0]):
    scores = pd.Series( X.T.iloc[:,i], index=adata.var_names )
    neg_score = np.quantile(scores, .005)
    pos_score = np.quantile(scores, .995)
    component_pos = X.T[i][ X.T[i]>pos_score].index
    component_neg = X.T[i][ X.T[i]<neg_score].index
    adata.var['sda_pos_high_C'+str(i)] = [i in component_pos for i in adata.var_names]
    adata.var['sda_neg_high_C'+str(i)] = [i in component_neg for i in adata.var_names]
    neg_score = np.quantile(scores, .01)
    pos_score = np.quantile(scores, .99)
    component_pos = X.T[i][ X.T[i]>pos_score].index
    component_neg = X.T[i][ X.T[i]<neg_score].index
    adata.var['sda_pos_low_C'+str(i)] = [i in component_pos for i in adata.var_names]
    adata.var['sda_neg_low_C'+str(i)] = [i in component_neg for i in adata.var_names]

In [16]:
adata.write('../write/SDA_object.h5ad')

In [22]:
adata

AnnData object with n_obs × n_vars = 10522 × 14409
    obs: 'G2M_score', 'S_score', 'batch', 'batch_single', 'cluster', 'condition', 'doublet_scores', 'dpt_pseudotime', 'initial_size', 'initial_size_spliced', 'initial_size_unspliced', 'leiden', 'leiden_sub', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'n_genes_by_counts', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'pct_counts_in_top_50_genes', 'percent_mito', 'phase', 'predicted_doublets', 'prop_spl', 'prop_unspl', 'recluster', 'scaled_CDR', 'total_counts', 'dpt_cellalign', 'SDA_cellscore_C0', 'SDA_cellscore_C1', 'SDA_cellscore_C2', 'SDA_cellscore_C3', 'SDA_cellscore_C4', 'SDA_cellscore_C5', 'SDA_cellscore_C6', 'SDA_cellscore_C7', 'SDA_cellscore_C8', 'SDA_cellscore_C9', 'SDA_cellscore_C10', 'SDA_cellscore_C11', 'SDA_cellscore_C12', 'SDA_cellscore_C13', 'SDA_cellscore_C14', 'sda_cell_relevant_C0', 'sda_cell_relevant_C1', 'sda_cell_relevant_C2', 'sda_cell_relev